# Pre-operative valve durability data preparation

**Clinical question**

> Given a patient's full pre-operative laboratory and medication trajectory, prior valve history extracted by Qwen, and the characteristics of a candidate valve being considered now, what is the model-estimated event-free durability of that candidate valve?

This notebook prepares the **real-data side** of that architecture.

Key rules:

- Labs and medications remain longitudinal trajectories.
- All historical valve facts come from the validated Qwen extraction.
- The valve being considered for the new procedure is a separate **candidate-valve input** supplied at inference time.
- For pre-operative prediction, no data occurring after the decision/implant time may enter the trajectory encoder.
- The 17 real patients are used to estimate empirical distributions, missingness, and longitudinal variability.
- Patients with an evidence-supported implant year can also be used as real implant-aligned case studies. They are **not** used as a clinical validation cohort.

In [ ]:
# ============================================================
# REAL 17-PATIENT COHORT — LOAD LABS + MEDICATIONS
# ============================================================

from pathlib import Path
import re
import numpy as np
import pandas as pd

LABS_FILE = Path("labs_deidentified.xlsx")
MEDS_FILE = Path("medication_dataset_EDA.xlsx")

COHORT = [
    f"Patient_{i:03d}"
    for i in range(101, 118)
]

# ---------------------------
# Load
# ---------------------------

labs = pd.read_excel(LABS_FILE)

meds = pd.read_excel(
    MEDS_FILE,
    sheet_name="Dedup View",
)

# Clean patient IDs
labs["Patient"] = (
    labs["Patient"]
    .astype(str)
    .str.strip()
)

meds["Patient"] = (
    meds["Patient"]
    .astype(str)
    .str.strip()
)

# Keep only multimodal 17
labs17 = labs[
    labs["Patient"].isin(COHORT)
].copy()

meds17 = meds[
    meds["Patient"].isin(COHORT)
].copy()

print("Lab rows:", len(labs17))
print("Medication rows:", len(meds17))

print(
    "Lab patients:",
    labs17["Patient"].nunique()
)

print(
    "Medication patients:",
    meds17["Patient"].nunique()
)

assert set(COHORT) == set(labs17["Patient"].unique())
assert set(COHORT) == set(meds17["Patient"].unique())

print("\n✓ All 17 patients present in both modalities")

In [ ]:
# ============================================================
# LABS → PATIENT × YEAR × 30 CURATED LAB FEATURES
# ============================================================

labs_work = labs17.copy()

labs_work["Result Year"] = pd.to_numeric(
    labs_work["Result Date"],
    errors="coerce",
).astype("Int64")

labs_work["Numeric Value"] = pd.to_numeric(
    labs_work["Numeric Value"],
    errors="coerce",
)

for col in [
    "Lab Component Name",
    "Loinc Code",
    "String Value",
    "Unit",
]:
    labs_work[col] = (
        labs_work[col]
        .astype("string")
        .str.strip()
    )


# ------------------------------------------------------------
# Preserve censored values such as >60 or <0.010
# ------------------------------------------------------------

CENSOR_RE = re.compile(
    r"^\s*([<>])\s*=?\s*"
    r"([-+]?\d*\.?\d+(?:[eE][-+]?\d+)?)\s*$"
)


def parse_censored_numeric(value):

    if pd.isna(value):
        return 0, np.nan

    m = CENSOR_RE.match(str(value))

    if not m:
        return 0, np.nan

    code = 1 if m.group(1) == ">" else -1
    threshold = float(m.group(2))

    return code, threshold


parsed = labs_work["String Value"].apply(
    parse_censored_numeric
)

labs_work["Censor_Code"] = (
    parsed
    .map(lambda x: x[0])
    .astype("int8")
)

labs_work["Censor_Threshold"] = (
    parsed
    .map(lambda x: x[1])
    .astype(float)
)

labs_work["Effective_Numeric"] = (
    labs_work["Numeric Value"]
)

use_threshold = (
    labs_work["Effective_Numeric"].isna()
    &
    labs_work["Censor_Threshold"].notna()
)

labs_work.loc[
    use_threshold,
    "Effective_Numeric",
] = labs_work.loc[
    use_threshold,
    "Censor_Threshold",
]


# ------------------------------------------------------------
# EXACT 30-feature specification used by synthetic/model pipeline
# ------------------------------------------------------------

LAB_FEATURES = [
    ("Hemoglobin", "loinc", ["718-7"]),
    ("Hematocrit", "loinc", ["4544-3"]),
    ("RBC", "loinc", ["789-8"]),
    ("WBC", "loinc", ["6690-2"]),
    ("Platelets", "loinc", ["777-3"]),
    ("MCV", "loinc", ["787-2"]),
    ("MCH", "loinc", ["785-6"]),
    ("MCHC", "raw_name", ["MCHC"]),
    ("RDW", "loinc", ["788-0"]),
    ("Glucose", "loinc", ["2339-0"]),
    ("Sodium", "loinc", ["2951-2"]),
    ("Potassium", "loinc", ["2823-3"]),
    ("Chloride", "loinc", ["2075-0"]),
    ("CO2", "loinc", ["2028-9"]),
    ("Calcium", "loinc", ["17861-6"]),
    ("Anion Gap", "loinc", ["33037-3"]),
    ("Creatinine", "loinc", ["2160-0"]),
    ("BUN", "loinc", ["3094-0"]),
    ("eGFR", "raw_name", ["eGFR-All Other Races"]),
    ("Albumin", "loinc", ["1751-7"]),
    ("Total Protein", "loinc", ["2885-2"]),
    ("AST", "loinc", ["1920-8"]),
    ("ALT", "loinc", ["1742-6"]),
    ("Alkaline Phosphatase", "loinc", ["6768-6"]),
    ("Total Bilirubin", "loinc", ["1975-2"]),
    ("INR", "raw_name", ["PT INR", "INR", "INR (POCT)"]),
    ("APTT", "loinc", ["14979-9"]),
    ("LVEF", "raw_name", ["LV Ejection Fraction"]),
    ("Troponin T", "raw_name", ["Troponin T"]),
    ("NT-proBNP", "raw_name", ["NT Pro BNP"]),
]

assert len(LAB_FEATURES) == 30


# ------------------------------------------------------------
# Select relevant raw lab observations
# ------------------------------------------------------------

lab_parts = []

for feature, selector_type, selectors in LAB_FEATURES:

    if selector_type == "loinc":

        subset = labs_work[
            labs_work["Loinc Code"].isin(selectors)
        ].copy()

    else:

        subset = labs_work[
            labs_work["Lab Component Name"].isin(selectors)
        ].copy()

    subset["Feature"] = feature

    lab_parts.append(subset)


lab_selected = pd.concat(
    lab_parts,
    ignore_index=True,
)

lab_selected = lab_selected.dropna(
    subset=[
        "Patient",
        "Result Year",
    ]
)


# ------------------------------------------------------------
# Median within Patient × Year
# ------------------------------------------------------------

lab_year_long = (
    lab_selected
    .groupby(
        [
            "Patient",
            "Result Year",
            "Feature",
        ],
        as_index=False,
    )
    .agg(
        lab_median=(
            "Effective_Numeric",
            "median",
        ),
        lab_n_obs=(
            "Effective_Numeric",
            lambda s: int(s.notna().sum()),
        ),
    )
)


lab_year = (
    lab_year_long
    .pivot_table(
        index=[
            "Patient",
            "Result Year",
        ],
        columns="Feature",
        values="lab_median",
        aggfunc="first",
    )
    .reset_index()
    .rename(
        columns={
            "Result Year": "Year"
        }
    )
)


# Guarantee every one of our 30 channels exists
for feature, _, _ in LAB_FEATURES:

    col = f"lab__{feature}"

    if feature in lab_year.columns:
        lab_year = lab_year.rename(
            columns={
                feature: col
            }
        )

    elif col not in lab_year.columns:
        lab_year[col] = np.nan


LAB_COLS = [
    f"lab__{feature}"
    for feature, _, _ in LAB_FEATURES
]

lab_year["labs_observed"] = 1


# canonical order
lab_year = lab_year[
    [
        "Patient",
        "Year",
        "labs_observed",
    ]
    + LAB_COLS
]


print(
    "Patient-year lab rows:",
    len(lab_year),
)

print(
    "Lab channels:",
    len(LAB_COLS),
)

display(
    lab_year.head(10)
)

In [ ]:
# ============================================================
# MEDICATIONS → PATIENT × YEAR × 15 MEDICATION CLASSES
# ============================================================

med = meds17.copy()


# ------------------------------------------------------------
# Dates
# ------------------------------------------------------------

for col in [
    "Start Date",
    "Administration Date",
    "End Date",
    "Discontinued Date",
]:

    med[col] = pd.to_numeric(
        med[col],
        errors="coerce",
    ).astype("Int64")


med["Row Multiplicity"] = pd.to_numeric(
    med["Row Multiplicity"],
    errors="coerce",
).fillna(1).clip(lower=1).astype(int)


for col in [
    "Simple Generic Name",
    "Medication Therapeutic Class",
    "Medication Pharmaceutical Class",
    "Medication Pharmaceutical Subclass",
    "Mode",
]:

    med[col] = (
        med[col]
        .astype("string")
        .str.strip()
    )


# ------------------------------------------------------------
# Observation year
#
# inpatient  -> administration year if available
# outpatient -> start year
# ------------------------------------------------------------

med["Medication Year"] = (
    med["Start Date"].copy()
)

inpatient_admin = (
    med["Mode"]
    .astype(str)
    .str.casefold()
    .eq("inpatient")
    &
    med["Administration Date"].notna()
)

med.loc[
    inpatient_admin,
    "Medication Year",
] = med.loc[
    inpatient_admin,
    "Administration Date",
]


# ------------------------------------------------------------
# Search across multiple medication descriptors
# ------------------------------------------------------------

generic = (
    med["Simple Generic Name"]
    .fillna("")
    .str.lower()
)

therapeutic = (
    med["Medication Therapeutic Class"]
    .fillna("")
    .str.lower()
)

pharm = (
    med["Medication Pharmaceutical Class"]
    .fillna("")
    .str.lower()
)

subclass = (
    med["Medication Pharmaceutical Subclass"]
    .fillna("")
    .str.lower()
)

med_search = (
    generic
    + " | "
    + therapeutic
    + " | "
    + pharm
    + " | "
    + subclass
)


# ------------------------------------------------------------
# SAME 15 groups used by model
# ------------------------------------------------------------

MED_GROUPS = {

    "beta_blocker":
        r"\bmetoprolol\b|\bcarvedilol\b|\blabetalol\b|"
        r"\batenolol\b|\bbisoprolol\b|\bpropranolol\b",

    "ace_inhibitor":
        r"\blisinopril\b|\benalapril\b|\bramipril\b|"
        r"\bbenazepril\b|\bcaptopril\b|\bquinapril\b",

    "arb":
        r"\bvalsartan\b|\blosartan\b|\bcandesartan\b|"
        r"\birbesartan\b|\bolmesartan\b|\btelmisartan\b",

    "arni":
        r"sacubitril",

    "loop_diuretic":
        r"\bfurosemide\b|\bbumetanide\b|\btorsemide\b",

    "thiazide_diuretic":
        r"hydrochlorothiazide|\bchlorthalidone\b|\bmetolazone\b",

    "mra":
        r"\bspironolactone\b|\beplerenone\b",

    "anticoagulant":
        r"\bwarfarin\b|\bapixaban\b|\brivaroxaban\b|"
        r"\bdabigatran\b|\bedoxaban\b|heparin|enoxaparin",

    "antiplatelet":
        r"\baspirin\b|\bclopidogrel\b|\bprasugrel\b|\bticagrelor\b",

    "statin":
        r"\batorvastatin\b|\bsimvastatin\b|\brosuvastatin\b|"
        r"\bpravastatin\b|\blovastatin\b",

    "ccb":
        r"\bamlodipine\b|\bdiltiazem\b|\bverapamil\b|\bnifedipine\b",

    "sglt2_inhibitor":
        r"\bempagliflozin\b|\bdapagliflozin\b|"
        r"\bcanagliflozin\b|\bertugliflozin\b",

    "digoxin":
        r"\bdigoxin\b",

    "antiarrhythmic":
        r"\bamiodarone\b|\bsotalol\b|\bdofetilide\b|"
        r"\bflecainide\b|\bpropafenone\b",

    "insulin":
        r"\binsulin\b",
}


for group, pattern in MED_GROUPS.items():

    med[f"grp__{group}"] = (
        med_search
        .str.contains(
            pattern,
            regex=True,
            na=False,
            case=False,
        )
        .astype("uint8")
    )


# ------------------------------------------------------------
# Patient × year aggregation
# ------------------------------------------------------------

med_year_source = med.dropna(
    subset=[
        "Patient",
        "Medication Year",
    ]
).copy()


def nunique_nonmissing(series):
    return int(
        series.dropna().nunique()
    )


med_year = (
    med_year_source
    .groupby(
        [
            "Patient",
            "Medication Year",
        ],
        as_index=False,
    )
    .agg(
        med_rows=(
            "Patient",
            "size",
        ),

        med_weighted_records=(
            "Row Multiplicity",
            "sum",
        ),

        med_unique_generic=(
            "Simple Generic Name",
            nunique_nonmissing,
        ),

        med_unique_therapeutic_classes=(
            "Medication Therapeutic Class",
            nunique_nonmissing,
        ),
    )
)


MED_COLS = []

for group in MED_GROUPS:

    source_col = f"grp__{group}"
    feature_col = f"med__{group}__present"

    MED_COLS.append(feature_col)

    presence = (
        med_year_source
        .groupby(
            [
                "Patient",
                "Medication Year",
            ]
        )[source_col]
        .max()
        .rename(feature_col)
        .reset_index()
    )

    med_year = med_year.merge(
        presence,
        on=[
            "Patient",
            "Medication Year",
        ],
        how="left",
    )


med_year = med_year.rename(
    columns={
        "Medication Year": "Year"
    }
)

med_year["medications_observed"] = 1


print(
    "Patient-year medication rows:",
    len(med_year),
)

print(
    "Medication channels:",
    len(MED_COLS),
)

display(
    med_year.head(10)
)

In [ ]:
# ============================================================
# REAL 17-PATIENT LAB + MEDICATION TIMELINE
# ============================================================

real_17_patient_year = (
    lab_year
    .merge(
        med_year,
        on=[
            "Patient",
            "Year",
        ],
        how="outer",
    )
    .sort_values(
        [
            "Patient",
            "Year",
        ]
    )
    .reset_index(drop=True)
)


# Observation flags:
# 0 = modality not observed in that patient-year
# 1 = modality observed
real_17_patient_year[
    "labs_observed"
] = (
    real_17_patient_year[
        "labs_observed"
    ]
    .fillna(0)
    .astype("uint8")
)

real_17_patient_year[
    "medications_observed"
] = (
    real_17_patient_year[
        "medications_observed"
    ]
    .fillna(0)
    .astype("uint8")
)


# IMPORTANT:
# Do NOT fill missing lab / med feature values here.
#
# NaN means modality/feature unobserved.
# The encoder later constructs the masks correctly.


print(
    "Rows:",
    len(real_17_patient_year)
)

print(
    "Patients:",
    real_17_patient_year["Patient"].nunique()
)

print(
    "Year range:",
    real_17_patient_year["Year"].min(),
    "→",
    real_17_patient_year["Year"].max(),
)

print(
    "Lab features:",
    len(LAB_COLS)
)

print(
    "Medication features:",
    len(MED_COLS)
)


display(
    real_17_patient_year[
        [
            "Patient",
            "Year",
            "labs_observed",
            "medications_observed",
            "lab__Creatinine",
            "lab__Hemoglobin",
            "lab__LVEF",
            "lab__NT-proBNP",
            "med__beta_blocker__present",
            "med__loop_diuretic__present",
            "med__anticoagulant__present",
            "med__antiplatelet__present",
        ]
    ].head(30)
)

In [ ]:
# ============================================================
# SAVE STAGE 1 REAL DATA
# ============================================================

OUT = Path(
    "real_17_patient_year_lab_med.csv"
)

real_17_patient_year.to_csv(
    OUT,
    index=False,
)

print(
    "✓ Saved:",
    OUT.resolve()
)

## Qwen valve history

This section deliberately keeps **historical valve information** separate from the proposed candidate valve.

The Qwen table may contain list-valued columns stored as strings in CSV. We parse those safely and retain the full extracted valve history. No later failure/reintervention field is used as a predictor of the proposed valve's durability.

In [ ]:
# ============================================================
# QWEN PRIOR VALVE HISTORY
# ============================================================

from pathlib import Path
import ast
import json
import numpy as np
import pandas as pd

VALVE_FILE = Path("valve_patient_data_17.csv")
valves = pd.read_csv(VALVE_FILE)

print("Valve patients:", valves["Patient"].nunique())
print("Available columns:")
print(valves.columns.tolist())


def safe_list(x):
    """Parse list-like CSV cells without executing arbitrary code."""
    if isinstance(x, list):
        return x
    if pd.isna(x):
        return []
    try:
        out = ast.literal_eval(str(x))
        return out if isinstance(out, list) else []
    except Exception:
        return []


LIST_COLUMNS = [
    "procedure_types",
    "procedure_years",
    "valve_models",
    "valve_sizes_mm",
    "redo_flags",
    "ViV_flags",
    "event_types",
    "event_years",
]

for col in LIST_COLUMNS:
    if col in valves.columns:
        valves[col] = valves[col].apply(safe_list)


# A compact history table.  Keep outcome/event fields for annotation/audit,
# but DO NOT feed them to a pre-operative prediction model unless they
# occurred before the new decision point.
history_cols = [
    c for c in [
        "Patient",
        "procedure_types",
        "procedure_years",
        "valve_models",
        "valve_sizes_mm",
        "redo_flags",
        "ViV_flags",
        "event_types",
        "event_years",
        "first_procedure_type",
        "first_procedure_year",
        "first_valve_model",
        "first_valve_size_mm",
        "latest_procedure_type",
        "latest_procedure_year",
        "latest_valve_model",
        "latest_valve_size_mm",
    ]
    if c in valves.columns
]

qwen_history = valves[history_cols].copy()

for col in ["first_procedure_year", "latest_procedure_year"]:
    if col in qwen_history.columns:
        qwen_history[col] = pd.to_numeric(
            qwen_history[col], errors="coerce"
        )

display(qwen_history.sort_values("Patient"))

## Preserve the complete longitudinal patient history

For the GRU encoders, calendar year is not itself the clinical target. We therefore add a universal chronological channel (`time_since_first_observation_months`) that is available for all 17 patients.

For the subset with a trustworthy Qwen implant year, we additionally compute implant-relative time. This is useful for real case studies and generator sanity checks, but it is **not required** for the other 14 patients to contribute distributional information.

In [ ]:
# ============================================================
# COMPLETE REAL LONGITUDINAL TABLE
# ============================================================

real_timeline = real_17_patient_year.copy()

real_timeline["Year"] = pd.to_numeric(
    real_timeline["Year"], errors="coerce"
)

first_observed_year = (
    real_timeline
    .groupby("Patient")["Year"]
    .transform("min")
)

real_timeline["time_since_first_observation_months"] = (
    real_timeline["Year"] - first_observed_year
) * 12.0


# Attach only the Qwen history metadata needed for alignment/audit.
anchor_cols = [
    c for c in [
        "Patient",
        "first_procedure_type",
        "first_procedure_year",
        "first_valve_model",
        "first_valve_size_mm",
    ]
    if c in qwen_history.columns
]

real_timeline = real_timeline.merge(
    qwen_history[anchor_cols],
    on="Patient",
    how="left",
    validate="many_to_one",
)

if "first_procedure_year" in real_timeline.columns:
    real_timeline["first_procedure_year"] = pd.to_numeric(
        real_timeline["first_procedure_year"],
        errors="coerce",
    )

    real_timeline["time_from_first_qwen_implant_months"] = (
        real_timeline["Year"]
        - real_timeline["first_procedure_year"]
    ) * 12.0

    real_timeline["qwen_implant_year_known"] = (
        real_timeline["first_procedure_year"].notna().astype("uint8")
    )
else:
    real_timeline["time_from_first_qwen_implant_months"] = np.nan
    real_timeline["qwen_implant_year_known"] = 0


print("Rows:", len(real_timeline))
print("Patients:", real_timeline["Patient"].nunique())
print(
    "Patients with Qwen implant year:",
    real_timeline.loc[
        real_timeline["qwen_implant_year_known"].eq(1),
        "Patient"
    ].nunique()
)

display(
    real_timeline[
        [
            "Patient",
            "Year",
            "time_since_first_observation_months",
            "first_procedure_type",
            "first_procedure_year",
            "time_from_first_qwen_implant_months",
            "labs_observed",
            "medications_observed",
        ]
    ].head(30)
)

## Empirical calibration from all 17 patients

The synthetic generator should **not** become a noisy copier of the three implant-anchored patients. Instead, all 17 real patients inform:

- lab location and spread,
- observed/missing rates,
- within-patient longitudinal changes,
- medication prevalence when observed,
- medication observation rates,
- medication start/stop transition frequencies.

These summaries are inputs to the synthetic generator; they are not estimates of clinical causality.

In [ ]:
# ============================================================
# EMPIRICAL LAB / MED TRAJECTORY STATISTICS — ALL 17
# ============================================================

def robust_iqr(s):
    s = pd.to_numeric(s, errors="coerce").dropna()
    if len(s) == 0:
        return np.nan
    return float(s.quantile(0.75) - s.quantile(0.25))


# ---------------------------
# LAB LEVEL + MISSINGNESS
# ---------------------------

lab_stats_rows = []

for col in LAB_COLS:
    s = pd.to_numeric(real_timeline[col], errors="coerce")
    obs = s.dropna()

    # Within-patient change between consecutive observed calendar rows.
    diffs = (
        real_timeline[["Patient", "Year", col]]
        .sort_values(["Patient", "Year"])
        .assign(
            value=lambda d: pd.to_numeric(d[col], errors="coerce")
        )
        .groupby("Patient")["value"]
        .diff()
        .dropna()
    )

    lab_stats_rows.append({
        "feature": col,
        "median": float(obs.median()) if len(obs) else np.nan,
        "iqr": robust_iqr(obs),
        "min": float(obs.min()) if len(obs) else np.nan,
        "max": float(obs.max()) if len(obs) else np.nan,
        "observed_rate": float(s.notna().mean()),
        "delta_median": float(diffs.median()) if len(diffs) else np.nan,
        "delta_iqr": robust_iqr(diffs),
        "delta_sd": float(diffs.std()) if len(diffs) > 1 else np.nan,
    })

lab_empirical_stats = pd.DataFrame(lab_stats_rows)


# ---------------------------
# MED PREVALENCE + TRANSITIONS
# ---------------------------

med_stats_rows = []

for col in MED_COLS:
    s = pd.to_numeric(real_timeline[col], errors="coerce")
    obs = s.dropna()

    transitions_01 = []
    transitions_10 = []
    transitions_same = []

    for _, g in (
        real_timeline[["Patient", "Year", col]]
        .sort_values(["Patient", "Year"])
        .groupby("Patient")
    ):
        vals = pd.to_numeric(g[col], errors="coerce")
        vals = vals.dropna().astype(int).to_numpy()

        if len(vals) < 2:
            continue

        prev = vals[:-1]
        nxt = vals[1:]

        transitions_01.extend(((prev == 0) & (nxt == 1)).tolist())
        transitions_10.extend(((prev == 1) & (nxt == 0)).tolist())
        transitions_same.extend((prev == nxt).tolist())

    med_stats_rows.append({
        "feature": col,
        "prevalence_when_observed": (
            float(obs.mean()) if len(obs) else np.nan
        ),
        "observed_rate": float(s.notna().mean()),
        "transition_0_to_1_rate": (
            float(np.mean(transitions_01))
            if transitions_01 else np.nan
        ),
        "transition_1_to_0_rate": (
            float(np.mean(transitions_10))
            if transitions_10 else np.nan
        ),
        "transition_same_rate": (
            float(np.mean(transitions_same))
            if transitions_same else np.nan
        ),
    })

med_empirical_stats = pd.DataFrame(med_stats_rows)

print("Lab features:", len(lab_empirical_stats))
print("Medication features:", len(med_empirical_stats))

display(lab_empirical_stats.head(10))
display(med_empirical_stats)

## Real implant-anchored case studies

For patients whose Qwen extraction provides an evidence-supported first implant year, we build a **strict pre-operative trajectory** using only years before the implant year.

Because the real table is annual rather than day-level, rows from the implant calendar year are excluded from the pre-op encoder to avoid accidentally including post-procedure measurements.

These cases are useful as real proof-of-concept examples, not as a clinical performance estimate.

In [ ]:
# ============================================================
# REAL PRE-OPERATIVE CASE-STUDY TRAJECTORIES
# ============================================================

anchored_patients = (
    qwen_history.loc[
        qwen_history["first_procedure_year"].notna(),
        "Patient"
    ]
    .drop_duplicates()
    .tolist()
    if "first_procedure_year" in qwen_history.columns
    else []
)

real_preop_anchored = real_timeline[
    real_timeline["Patient"].isin(anchored_patients)
].copy()

# Strictly pre-implant because our real time resolution is only yearly.
real_preop_anchored = real_preop_anchored[
    real_preop_anchored["Year"]
    < real_preop_anchored["first_procedure_year"]
].copy()

real_preop_anchored["months_before_implant"] = (
    real_preop_anchored["first_procedure_year"]
    - real_preop_anchored["Year"]
) * 12.0

real_preop_anchored = (
    real_preop_anchored
    .sort_values(["Patient", "Year"])
    .reset_index(drop=True)
)

coverage = (
    real_preop_anchored
    .groupby("Patient")
    .agg(
        implant_year=("first_procedure_year", "first"),
        implanted_procedure=("first_procedure_type", "first"),
        implanted_model=("first_valve_model", "first"),
        implanted_size_mm=("first_valve_size_mm", "first"),
        first_history_year=("Year", "min"),
        last_preop_year=("Year", "max"),
        n_preop_timepoints=("Year", "size"),
        n_lab_years=("labs_observed", "sum"),
        n_med_years=("medications_observed", "sum"),
    )
    .reset_index()
)

print("Anchored patients:", len(anchored_patients))
print(
    "Anchored patients with at least one strict pre-op row:",
    real_preop_anchored["Patient"].nunique()
)

display(coverage)

## Candidate valve = physician-controlled scenario input

The proposed new valve must **not** be inferred from the patient's future record. At inference time it is a scenario variable supplied by the clinician.

The same patient representation can therefore be evaluated repeatedly while changing only the proposed valve. This is what enables *patient-specific durability estimates for multiple candidate valves*.

The helper below creates scenario rows. The model will later evaluate each row with the same patient embedding.

In [ ]:
# ============================================================
# CANDIDATE-VALVE SCENARIO BUILDER
# ============================================================

CANDIDATE_VALVE_FIELDS = [
    "candidate_procedure_type",
    "candidate_valve_model",
    "candidate_valve_size_mm",
    "candidate_valve_position",
]


def make_candidate_valve_scenarios(
    patient_id,
    candidates,
):
    """
    candidates: list of dicts, e.g.
    [
        {
            "candidate_procedure_type": "TAVR",
            "candidate_valve_model": "Candidate_A",
            "candidate_valve_size_mm": 26,
            "candidate_valve_position": "aortic",
        },
        ...
    ]

    Returns one row per proposed valve.  Patient history is NOT changed.
    """
    if patient_id not in set(real_timeline["Patient"]):
        raise ValueError(f"Unknown patient: {patient_id}")

    rows = []

    for i, candidate in enumerate(candidates):
        row = {
            "Patient": patient_id,
            "candidate_id": f"candidate_{i+1:02d}",
        }

        for field in CANDIDATE_VALVE_FIELDS:
            row[field] = candidate.get(field, np.nan)

        rows.append(row)

    return pd.DataFrame(rows)


# Example only — replace with valves the clinician wants to compare.
example_candidates = [
    {
        "candidate_procedure_type": "TAVR",
        "candidate_valve_model": "Candidate_A",
        "candidate_valve_size_mm": 26,
        "candidate_valve_position": "aortic",
    },
    {
        "candidate_procedure_type": "TAVR",
        "candidate_valve_model": "Candidate_B",
        "candidate_valve_size_mm": 29,
        "candidate_valve_position": "aortic",
    },
]

if len(COHORT):
    display(
        make_candidate_valve_scenarios(
            COHORT[0],
            example_candidates,
        )
    )

## Export contract for the synthetic generator and model

The next notebook should consume:

1. `real_17_patient_year_lab_med.csv` — full real trajectories from all 17 patients.
2. `real_lab_empirical_stats.csv` — lab distributions, missingness and longitudinal change statistics.
3. `real_med_empirical_stats.csv` — medication prevalence, missingness and transitions.
4. `qwen_prior_valve_history_17.pkl` — Qwen-derived historical valve information.
5. `real_preop_anchored_case_studies.csv` — strict pre-op trajectories for the small implant-anchored real subset.

### Model target

For each synthetic patient and proposed valve \(v\):

\[
S(t \mid H_{pre}, V_{history}, v)
=
P(T_{valve} > t)
\]

where:

- \(H_{pre}\): pre-operative lab and medication history,
- \(V_{history}\): prior valve history extracted by Qwen,
- \(v\): proposed candidate valve,
- \(T_{valve}\): time from implantation to clinically significant valve failure/reintervention.

The UI can then report median predicted durability, restricted mean event-free years, and 2/5/8/10-year event-free probabilities for each candidate valve.

In [ ]:
# ============================================================
# SAVE PRE-OP DECISION-SUPPORT INPUTS
# ============================================================

real_timeline.to_csv(
    "real_17_patient_year_lab_med_with_qwen_anchor.csv",
    index=False,
)

lab_empirical_stats.to_csv(
    "real_lab_empirical_stats.csv",
    index=False,
)

med_empirical_stats.to_csv(
    "real_med_empirical_stats.csv",
    index=False,
)

qwen_history.to_pickle(
    "qwen_prior_valve_history_17.pkl"
)

real_preop_anchored.to_csv(
    "real_preop_anchored_case_studies.csv",
    index=False,
)

coverage.to_csv(
    "real_preop_anchored_coverage.csv",
    index=False,
)

print("✓ Saved real_17_patient_year_lab_med_with_qwen_anchor.csv")
print("✓ Saved real_lab_empirical_stats.csv")
print("✓ Saved real_med_empirical_stats.csv")
print("✓ Saved qwen_prior_valve_history_17.pkl")
print("✓ Saved real_preop_anchored_case_studies.csv")
print("✓ Saved real_preop_anchored_coverage.csv")